In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from collections import Counter
from edc_pdutils.dataframes import get_crf
from intecomm_analytics.dataframes import get_df_main_for_crfs, get_hiv_rx_crf, get_htn_rx_crf, get_dm_rx_crf
from intecomm_analytics.dataframes.main_1858_to_stata import df_main_variable_labels
from intecomm_analytics.dataframes.main_1858_to_stata import to_stata
from intecomm_subject.models import ComplicationsFollowup, ComplicationsBaseline
from django_pandas.io import read_frame
from edc_analytics.stata import get_stata_labels_from_model


In [ ]:
def get_model_and_merge_with_df_main(df:pd.DataFrame, model:str|None, suffix:str, drop_cols:list[str]|None=None, df_crf:pd.DataFrame|None=None)->pd.DataFrame:
    drop_cols = drop_cols or []
    _, model_name = model.split(".")
    system_columns = ["id", "consent_model", "consent_version", "crf_status", "crf_status_comments", "created", "modified", "user_created", "user_modified", "hostname_created", "hostname_modified", "device_created", "device_modified", "locale_created", "locale_modified", "revision"]
    visit_columns = []

    if not isinstance(df_crf, pd.DataFrame):
        df_crf = get_crf(model, subject_visit_model="intecomm_subject.subjectvisit", read_verbose=False).drop(columns=drop_cols)
    df_crf = (
        df_crf[[col for col in df_crf.columns if col not in visit_columns and col not in system_columns]]
        .copy()
        .rename(columns={col:f"{col}__{suffix}" for col in df_crf.columns if col not in ["subject_visit_id", ]})
    )
    df_crf[f"crf_{model_name}"] = 1

    # merge
    df = df.merge(df_crf, on="subject_visit_id", how="left", suffixes=("", f"__{suffix}"))

    # get rid of duplicate visit cols from the merge with crf
    list_with_dups = [col.split(f"__{suffix}")[0] for col in df.columns if col.endswith(f"__{suffix}")]
    list_with_dups = list_with_dups + list(df.columns)
    counts = Counter(list_with_dups)
    duplicates = [item for item in list_with_dups if counts[item] > 1]
    duplicates = [f"{col}__{suffix}" for col in duplicates]
    df = df.drop(columns=duplicates).drop(columns=[f"appointment_id__{suffix}"])
    old = [col for col in df.columns if f"__{suffix}" in col]
    new = [col.replace(f"__{suffix}", f"_{suffix}") for col in df.columns if f"__{suffix}" in col]
    df = df.rename(columns=dict(zip(old, new)))
    return df

In [ ]:
variable_labels = {}

df_main_orig = get_df_main_for_crfs(fasting_hours=8.0)

In [ ]:
df_main = df_main_orig.copy()

In [ ]:

# eq5d3l
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.eq5d3l", "eq5d3l")
variable_labels.update(**get_stata_labels_from_model(df_main, "intecomm_subject.eq5d3l", "eq5d3l"))

In [ ]:
# icecapa
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.icecapa", "icecapa")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.icecapa", "icecapa"))

In [ ]:
# assets
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicsassets", "hhassets")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.healtheconomicsassets", "hhassets"))

In [ ]:
# csa
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.careseekinga", "csa")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.careseekinga", "csa"))

In [ ]:
# csb
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.careseekingb", "csb")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.careseekingb", "csb"))

In [ ]:
# hh
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicshouseholdhead", "hhhead")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.healtheconomicshouseholdhead", "hhhead"))

In [ ]:
# income
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicsincome", "hhincome")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.healtheconomicsincome", "hhincome"))

In [ ]:
# patient
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicspatient", "patient")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.healtheconomicspatient", "patient"))

In [ ]:
# property
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicsproperty", "hhproperty")
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.healtheconomicsproperty", "hhproperty"))

In [ ]:
# complications
df_crf1 = read_frame(ComplicationsBaseline.objects.all())
df_crf2 = read_frame(ComplicationsFollowup.objects.all())
df_crf = pd.concat([df_crf1, df_crf2])

df_crf = df_crf.rename(columns={
    "subject_visit": "subject_visit_id",
    "heart_attack": "complication_heart_attack",
    "renal_disease": "complication_renal_disease",
    "vision": "complication_vision",
    "numbness": "complication_numbness",
    "foot_ulcers": "complication_foot_ulcers",
    "stroke":"complication_stroke",
    "heart_attack_date": "complication_heart_attack_date",
    "renal_disease_date": "complication_renal_disease_date",
    "vision_date": "complication_vision_date",
    "numbness_date": "complication_numbness_date",
    "foot_ulcers_date": "complication_foot_ulcers_date",
    "stroke_date":"complication_stroke_date",
})
for col in [
    "complication_stroke_date",
    "complication_heart_attack_date",
    "complication_renal_disease_date",
    "complication_vision_date",
    "complication_numbness_date",
    "complication_foot_ulcers_date"
]:
    df_crf[col] = df_crf[col].astype("datetime64[ns]")

df_crf["crf_complications"] = 1

df_main = df_main.drop(columns=["complication_stroke",
    "complication_heart_attack",
    "complication_renal_disease",
    "complication_vision",
    "complication_numbness",
    "complication_foot_ulcers"])

df_main = df_main.merge(df_crf[[
    "subject_visit_id",
    "crf_complications",
    "complication_stroke",
    "complication_stroke_date",
    "complication_heart_attack",
    "complication_heart_attack_date",
    "complication_renal_disease",
    "complication_renal_disease_date",
    "complication_vision",
    "complication_vision_date",
    "complication_numbness",
    "complication_numbness_date",
    "complication_foot_ulcers",
    "complication_foot_ulcers_date"]], on="subject_visit_id", how="left")

variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.complicationsfollowup", "complication"))


In [ ]:
# missed visit
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.subjectvisitmissed", "mv")
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_item" in col])
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_identifier" in col])
variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.subjectvisitmissed", "mv"))


In [ ]:
df_crf = get_hiv_rx_crf()

suffix = "hiv"
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.drugrefillhiv", suffix, df_crf=df_crf)
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_item" in col])
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_identifier" in col])

old = [col for col in df_main.columns if col.endswith("_hiv")]
new = [col[:-4] for col in df_main.columns if col.endswith("_hiv")]
df_main = df_main.rename(columns=dict(zip(old, new)))
df_main = df_main.rename(columns={
    "modifications": f"{suffix}_rx_modifications",
    "modifications_other": f"{suffix}_rx_modifications_other",
    "modifications_reason": f"{suffix}_rx_modifications_reason",
    "modifications_reason_other": f"{suffix}_rx_modifications_reason_other",
})

variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.drugrefillhiv", suffix))
variable_labels.update({
    f"{suffix}_rx": "HIV regimens",
    f"{suffix}_rx_changed": "True if change between first and last rx, not considering interim reports",
    f"{suffix}_rx_first": f"First reported {suffix} medication",
    f"{suffix}_rx_last": f"Last reported {suffix} medication",
    f"{suffix}_rx_suppl": f"Supplemental meds reported on {suffix} medication report",
    f"{suffix}_rx_concomitant": f"Concomitant meds reported on {suffix} medication report"
})


In [ ]:
df_crf = get_htn_rx_crf()
suffix = "htn"
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.drugrefillhtn", suffix, df_crf=df_crf)
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_item" in col])
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_identifier" in col])

old = [col for col in df_main.columns if col.endswith("_htn")]
new = [col[:-4] for col in df_main.columns if col.endswith("_htn")]
df_main = df_main.rename(columns=dict(zip(old, new)))
df_main = df_main.rename(columns={
    "modifications": f"{suffix}_rx_modifications",
    "modifications_other": f"{suffix}_rx_modifications_other",
    "modifications_reason": f"{suffix}_rx_modifications_reason",
    "modifications_reason_other": f"{suffix}_rx_modifications_reason_other",
})

variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.drugrefillhtn", suffix))
variable_labels.update({
    f"{suffix}_rx": "HTN medications",
    f"{suffix}_rx_changed": "True if change between first and last rx, not considering interim reports",
    f"{suffix}_rx_first": f"First reported {suffix} medication",
    f"{suffix}_rx_last": f"Last reported {suffix} medication",
    f"{suffix}_rx_suppl": f"Supplemental meds reported on {suffix} medication report",
    f"{suffix}_rx_concomitant": f"Concomitant meds reported on {suffix} medication report",
    f"{suffix}_rx_days": f"Days prescribed counting from day of {suffix} medication report",
})


In [ ]:

df_crf = get_dm_rx_crf()
suffix = "dm"
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.drugrefilldm", suffix, df_crf=df_crf)
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_item" in col])
df_main = df_main.drop(columns=[col for col in df_main.columns if "action_identifier" in col])

old = [col for col in df_main.columns if col.endswith("_dm")]
new = [col[:-3] for col in df_main.columns if col.endswith("_dm")]
df_main = df_main.rename(columns=dict(zip(old, new)))
df_main = df_main.rename(columns={
    "modifications": f"{suffix}_rx_modifications",
    "modifications_other": f"{suffix}_rx_modifications_other",
    "modifications_reason": f"{suffix}_rx_modifications_reason",
    "modifications_reason_other": f"{suffix}_rx_modifications_reason_other",
})

variable_labels.update(**get_stata_labels_from_model(df_main,"intecomm_subject.drugrefilldm", suffix))
variable_labels.update({
    f"{suffix}_rx": "DM medications",
    f"{suffix}_rx_changed": "True if change between first and last rx, not considering interim reports",
    f"{suffix}_rx_first": f"First reported {suffix} medication",
    f"{suffix}_rx_last": f"Last reported {suffix} medication",
    f"{suffix}_rx_suppl": f"Supplemental meds reported on {suffix} medication report",
    f"{suffix}_rx_concomitant": f"Concomitant meds reported on {suffix} medication report",
    f"{suffix}_rx_days": f"Days prescribed counting from day of {suffix} medication report",
})


In [ ]:
df_main = df_main.drop(columns=[col for col in df_main.columns if col.startswith("glucose_fasting_duration")])

In [ ]:
# df_main[[col for col in df_main.columns if col.startswith("dm")]]

In [ ]:
# variable_labels

In [ ]:
# these are the long field names
# go through this by hand and shorten
original_labels = [
    "hiv_rx_modifications_reason_other",
    "htn_rx_modifications_reason_other",
    "dm_rx_modifications_reason_other",
    'primary_vl_controlled_baseline_400',
    'primary_vl_controlled_baseline_50',
    'primary_vl_controlled_endline_400',
    'health_today_score_confirmed_eq5d3l',
    'external_wall_material_other_hhassets',
    'external_window_material_hhassets',
    'external_window_material_other_hhassets',
    'med_not_collected_reason_other_csa',
    'med_not_collected_reason_other_csb',
    'inpatient_household_nowork_days_csb',
    'inpatient_money_sources_other_csb',
    'inpatient_money_sources_main_other_csb',
    'rental_income_value_known_hhincome',
    'ngo_assistance_value_known_hhincome',
    'internal_remit_value_known_hhincome',
    'external_remit_value_known_hhincome',
    'more_sources_value_known_hhincome',
    'external_remit_currency_other_hhincome',
    'financial_status_compare_hhincome',
    'pat_employment_type_other_patient',
    'land_surface_area_units_hhproperty',
    'calculated_land_surface_area_hhproperty'
]
shortened_labels = [
    "hiv_rx_mod_reason_other",
    "htn_rx_mod_reason_other",
    "dm_rx_mod_reason_other",
    'primary_vl_cntrl_baseline_400',
    'primary_vl_cntrl_baseline_50',
    'primary_vl_cntrl_endline_400',
    'health_today_score_conf_eq5d3l',
    'ext_wall_materl_other_hhassets',
    'ext_window_materl_hhassets',
    'ext_window_materl_other_hhassets',
    'med_not_collect_reason_other_csa',
    'med_not_collect_reason_other_csb',
    'inpatient_hh_nowork_days_csb',
    'inpatient_mny_src_other_csb',
    'inpatient_mny_src_main_other_csb',
    'rental_income_val_known_hhincome',
    'ngo_asst_val_known_hhincome',
    'int_remit_val_known_hhincome',
    'ext_remit_val_known_hhincome',
    'more_src_val_known_hhincome',
    'ext_remit_curr_other_hhincome',
    'financl_status_compare_hhincome',
    'pat_emply_type_other_patient',
    'land_surf_area_units_hhproperty',
    'calc_land_surf_area_hhproperty'
]

In [ ]:
# export
df_main["roof_material_other_hhassets"] = df_main["roof_material_other_hhassets"].fillna("")
df_main["tests_not_done_other_csa"] = df_main["tests_not_done_other_csa"].fillna("")
df_main["no_accessed_care_other_csb"] = df_main["no_accessed_care_other_csb"].fillna("")
df_main["hoh_education_other_hhhead"] = df_main["hoh_education_other_hhhead"].fillna("")
df_main["land_surface_area_hhproperty"] = df_main["land_surface_area_hhproperty"].astype("Float64")
df_main["calculated_land_surface_area_hhproperty"] = df_main["calculated_land_surface_area_hhproperty"].astype("Float64")


# rename cols in the Dataframe using shortened col names
rename_cols = dict(zip(original_labels, shortened_labels))
df_main = df_main.rename(columns=rename_cols)

In [ ]:
# update the variable labels for stata with the shortened col names
rev_variable_labels = {description: fld for fld, description in variable_labels.items()}
for orig_fld, shortened_fld in rename_cols.items():
    if orig_fld in variable_labels:
        rev_variable_labels[variable_labels[orig_fld]] = shortened_fld
variable_labels = {v:k for k,v in rev_variable_labels.items()}

In [ ]:
variable_labels.update(**df_main_variable_labels())

In [ ]:
df_main["weight"] = df_main["weight"].astype("Float64")
df_main["height"] = df_main["height"].astype("Float64")
df_main["bmi"] = df_main["bmi"].astype("Float64")
df_main["vl_baseline"] = df_main["vl_baseline"].astype("Float64")
df_main["vl_endline"] = df_main["vl_endline"].astype("Float64")
df_main["glucose_value_baseline"] = df_main["glucose_value_baseline"].astype("Float64")
df_main["glucose_value_endline"] = df_main["glucose_value_endline"].astype("Float64")
df_main["hiv_rx_modifications_other"] = df_main.hiv_rx_modifications_other.fillna("")
df_main["htn_rx_modifications_other"] = df_main.htn_rx_modifications_other.fillna("")
df_main["hiv_rx_changed"] = df_main["hiv_rx_changed"].astype("Int64")
df_main["htn_rx_changed"] = df_main["htn_rx_changed"].astype("Int64")
df_main["dm_rx_changed"] = df_main["dm_rx_changed"].astype("Int64")


In [ ]:
to_stata(df_main, analysis_folder, filename="df_he.dta", stata_labels=variable_labels)